# Train the LLM router on a Colab GPU

By default this trains on the **capability dataset** — name-free descriptions labelled by what kind of AI suits each prompt (clean labels, no 2025-era arena bias). The arena dataset remains available as an alternative.

A free T4 is plenty: the capability run (~3.4k pairs, 4 epochs) takes a couple of minutes; the arena run (~15k pairs, 3 epochs) about 10–15.

**Before you start:**
1. `Runtime → Change runtime type → T4 GPU → Save`.
2. Push your latest local code to GitHub, or the clone below gets the old scripts (the capability-dataset builder is new):
   ```
   git add -A && git commit -m "Capability dataset + Colab notebook" && git push
   ```

In [ ]:
# Should list an NVIDIA GPU. If not: Runtime > Change runtime type > GPU, then re-run.
!nvidia-smi

## 1. Get the code

In [ ]:
REPO = "/content/llm-router-project"
GITHUB_URL = "https://github.com/PolarDyth/llm-router-project.git"

import os
if not os.path.isdir(REPO):
    !git clone {GITHUB_URL} {REPO}

# No GitHub? Upload `training/train_router.py` and `device_utils.py` via the
# Files panel (folder icon on the left), then run:
#   !mkdir -p {REPO}/training
#   !mv /content/train_router.py {REPO}/training/
#   !mv /content/device_utils.py {REPO}/

## 2. Build the capability dataset

Upload `data/model_profiles.json` (~130 KB) from your Mac when prompted — the builder turns it into ~7,000 name-free training pairs: synthetic capability cards plus real OpenRouter descriptions, scrubbed of model names, codenames, version digits and suffix words.

*Optional:* also upload `router_training_pairs_scrubbed.csv` (~9 MB) into `{REPO}/data/` — the builder mines real user prompts from it (strictly keyword-labeled, prompts that mention models are skipped) to diversify training prompts.

*Alternative:* skip the build entirely and upload a prebuilt `router_training_pairs_capabilities.csv` into `{REPO}/data/`, then point `DATASET` at it in the training cell.


In [ ]:
import os, shutil
from google.colab import files

os.makedirs(f"{REPO}/data", exist_ok=True)
profiles_path = f"{REPO}/data/model_profiles.json"
if not os.path.exists(profiles_path):
    uploaded = files.upload()  # pick model_profiles.json
    for name in uploaded:
        shutil.move(name, f"{REPO}/data/{name}")

!python {REPO}/scripts/build_capability_dataset.py

## 3. Install dependencies

Colab already ships a CUDA-enabled torch, which satisfies `torch>=2.0`, so pip leaves it alone and only installs/updates the rest.

In [ ]:
!pip install -q -r {REPO}/requirements.txt

## 4. Mount Google Drive (recommended)

Free Colab VMs are wiped when the session ends — train straight to Drive so the model survives.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/llm-router/dynamic_router_final"
# Prefer the ephemeral VM instead? Use:
# OUTPUT_DIR = f"{REPO}/models/dynamic_router/final"

## 5. Train

In [ ]:
# Capability dataset (default): ~3.4k pairs, clean labels -> expect the loss to
# drop well below the arena run's ~0.68 plateau.
DATASET = f"{REPO}/data/router_training_pairs_capabilities.csv"
# Arena dataset instead? Upload router_training_pairs_scrubbed.csv into
# {REPO}/data/ and use --epochs 3 --warmup-steps 100:
# DATASET = f"{REPO}/data/router_training_pairs_scrubbed.csv"

!python {REPO}/training/train_router.py --dataset "{DATASET}" --output-dir "{OUTPUT_DIR}" --epochs 4 --batch-size 32 --warmup-steps 40 --device cuda

## 6. Get the model back (skip if you trained to Drive)

Zip and download, then unzip into `models/dynamic_router/final` in your local repo so `scripts/test_router.py` finds it.

In [ ]:
# from google.colab import files
# !zip -r /content/router_model.zip "{OUTPUT_DIR}"
# files.download("/content/router_model.zip")